<a href="https://colab.research.google.com/github/mayait/CursoAnalisisDatos_IA_2026/blob/main/sitio/labs/lab_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Laboratorio 1 · Introducción a Business Analytics en la era de la IA

No hay semana anterior de la que colgarse, así que empezamos por el final: hoy recorres un análisis
completo de principio a fin —cargar, describir, rankear, graficar, concluir— sobre el negocio más
fácil de imaginar que existe: **una pizzería**. Un año de tickets reales, 32 pizzas en el menú y la
pregunta que el dueño hace en voz alta: *«¿cuál es la pizza estrella y cuál eliminamos?»*. Vas a ver
que la respuesta cambia según cómo definas «estrella», y que ese cambio no es un tecnicismo: decide
qué pizza sale del menú. Las primeras cuatro semanas del curso entrenan con datos públicos de una
industria distinta cada una —hoy comida, luego retail, música y streaming—; no vienes a aprender
pandas, vienes a ver el arco entero para reconocer, en las quince semanas que siguen, dónde se pierde
un análisis. Esta sesión desbloquea las dos cosas que usarás todo el curso: un cuaderno que corre sin
instalar nada y un protocolo para encargarle trabajo a un asistente de IA sin quedar a merced de lo
que te responda.

> **Hoy haces** · Ejecutas un análisis completo sobre un año de ventas de una pizzería (90 min):
> cargas la tabla, describes lo que pasó, produces dos rankings y un gráfico y escribes una conclusión
> de negocio. Después reproduces un informe con datos correctos y conclusión inválida —el ranking
> engañoso— y lo corriges cambiando una sola cosa: la métrica. Cierras abriendo tu bitácora de prompts.
>
> **Entrega** · Este cuaderno ejecutado de arriba a abajo, con los tres ejercicios resueltos y al menos
> tres filas en la bitácora de prompts. Nombre de archivo: `lab_01_apellido.ipynb`.

In [ ]:
# --- Setup del entorno ---
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Los datasets públicos de las semanas 1 a 4 viven en sitio/datos/publicos/
CANDIDATOS = [Path("../datos/publicos"), Path("datos/publicos"), Path("sitio/datos/publicos")]
DATOS = next((p for p in CANDIDATOS if p.exists()), None)
if DATOS is None:
    raise FileNotFoundError(
        "No encuentro la carpeta de datos. En Colab ejecuta primero:\n"
        "  !git clone https://github.com/<usuario>/CursoAnalisisDatos_IA_2026.git")

print("Setup completo ✓")
print(f"pandas {pd.__version__} · datos en {DATOS.resolve()}")

## 1. Qué contesta cada peldaño de la analítica

Antes de tocar un archivo, la pregunta es qué tipo de respuesta necesitas. Los cuatro niveles no son
una escalera de dificultad técnica sino de compromiso: cada peldaño obliga a decidir algo más. Saltarse
un peldaño es el origen de la mitad de los proyectos de analítica que se abandonan: nadie predice bien
un negocio que todavía no sabe describir.

In [ ]:
escalera = pd.DataFrame([
    ("Descriptivo",  "¿Qué pasó?",             "Vendimos 49 574 pizzas en un año",                 "Reporte"),
    ("Diagnóstico",  "¿Por qué pasó?",         "La Brie Carre vende poco porque es la más cara",   "Hipótesis"),
    ("Predictivo",   "¿Qué va a pasar?",       "Este viernes saldrán unas 140 pizzas del horno",   "Prioridad"),
    ("Prescriptivo", "¿Qué hago al respecto?", "Retirar dos pizzas del menú y subir un precio",    "Acción"),
], columns=["nivel", "pregunta", "ejemplo en la pizzería", "qué habilita"])
escalera

Hoy te quedas en los dos primeros peldaños. Es deliberado: el descriptivo mal hecho contamina todo lo
que viene después, y es justo donde se rompe el informe que vas a reproducir.

## 2. Cargar: los primeros cinco minutos con una tabla nueva

El archivo es el registro público de un año completo de una pizzería real de Estados Unidos: cada
ticket, cada pizza, cada precio. Empieza por saber qué tamaño tiene lo que acabas de abrir.

In [ ]:
pizzas = pd.read_csv(DATOS / "pizza_ventas.csv")

print(f"pizzas : {pizzas.shape[0]:,} filas × {pizzas.shape[1]} columnas")
pizzas.head()

📌 Cuarenta y ocho mil seiscientas veinte filas. Ojo con lo que representa **una fila**: no es una
pizza ni un pedido, es una **línea de pedido** (una pizza concreta dentro de un ticket, con su
cantidad y su precio). Esa distinción parece un detalle y es el tema completo de la semana 2.

## 3. Describir: qué pasó

Cinco números y un rango de fechas. Con eso ya puedes hablar con el dueño: cuánto entró, en qué
periodo, cuántos tickets y cuántas pizzas hay detrás.

In [ ]:
pizzas["fecha"] = pd.to_datetime(pizzas["date"])

# El ingreso de una línea: cantidad × precio.
pizzas["ingreso"] = pizzas["quantity"] * pizzas["price"]

print(f"periodo             : {pizzas['fecha'].min():%d-%m-%Y} a {pizzas['fecha'].max():%d-%m-%Y}")
print(f"ingresos            : {pizzas['ingreso'].sum():,.2f}")
print(f"pedidos (tickets)   : {pizzas['Factura'].nunique():,}")
print(f"pizzas vendidas     : {pizzas['quantity'].sum():,}")
print(f"pizzas en el menú   : {pizzas['name'].nunique()}")
print(f"líneas de pedido    : {len(pizzas):,}")
print(f"pizzas por pedido   : {pizzas.groupby('Factura')['quantity'].sum().mean():.2f}")
print(f"ingreso por pedido  : {pizzas.groupby('Factura')['ingreso'].sum().mean():,.2f}")

Ochocientos diecisiete mil ochocientos sesenta dólares en un año, repartidos en 21 350 tickets y
49 574 pizzas de un menú de 32. El pedido típico lleva 2,32 pizzas y deja 38,31. Con esos números ya
describes el negocio. Lo que todavía no puedes hacer es contestar la pregunta que al dueño le importa.

## 4. El informe del administrador: la estrella por unidades

*«¿Cuál es la pizza estrella y cuál eliminamos?»* El administrador contestó con el ranking más
natural del mundo: unidades vendidas.

In [ ]:
por_unidades = pizzas.groupby("name")["quantity"].sum().sort_values(ascending=False)

print("LAS CINCO MÁS VENDIDAS (unidades)")
print(por_unidades.head(5).to_string())
print()
print("LAS CINCO MENOS VENDIDAS (unidades)")
print(por_unidades.tail(5).to_string())
print()
print(f"la primera vende {por_unidades.iloc[0] / por_unidades.iloc[-1]:.1f} veces más que la última")

Y este es el informe que recibió el dueño, textual:

> «La estrella es la **Classic Deluxe**, con 2 453 unidades en el año. La peor es la **Brie Carre**,
> con 490: la líder vende 5,0 veces más. Recomendamos retirar la Brie Carre del menú y dedicar la
> promoción del próximo trimestre a la Classic Deluxe.»

Los números son correctos. Puedes verificarlos tú mismo arriba. La conclusión, en cambio, no se
sostiene: **la pregunta era cuál pizza es más valiosa y lo que se midió fue cuál sale más veces del
horno.**

## 5. La corrección: mismo ranking, otra métrica

Una pizzería no deposita unidades en el banco: deposita dólares. La comparación honesta con la
pregunta del dueño multiplica cada línea por su precio.

In [ ]:
por_ingresos = pizzas.groupby("name")["ingreso"].sum().sort_values(ascending=False)

print("LAS CINCO QUE MÁS FACTURAN")
print(por_ingresos.head(5).to_string())
print()
puesto_thai_uds = list(por_unidades.index).index("The Thai Chicken Pizza") + 1
puesto_deluxe_ing = list(por_ingresos.index).index("The Classic Deluxe Pizza") + 1
print(f"estrella por unidades : {por_unidades.index[0]}  ({por_unidades.iloc[0]:,} uds)")
print(f"estrella por ingresos : {por_ingresos.index[0]}  ({por_ingresos.iloc[0]:,.2f})")
print(f"la Thai Chicken es {puesto_thai_uds}ª por unidades y 1ª por ingresos")
print(f"la Classic Deluxe cae al puesto {puesto_deluxe_ing} por ingresos ({por_ingresos['The Classic Deluxe Pizza']:,.2f})")
print(f"diferencia entre ambas: {(por_ingresos.iloc[0] / por_ingresos['The Classic Deluxe Pizza'] - 1) * 100:.1f} % a favor de la Thai Chicken")

El podio se reordena entero. La **Thai Chicken**, quinta por unidades con 2 371, es la número uno en
ingresos con 43 434,25: un 13,8 % más que la Classic Deluxe, que cae al cuarto puesto con 38 180,50.
Vende menos pizzas y factura más porque su precio medio está entre los más altos del menú. «La
estrella» depende de la definición de estrella, y esa definición es una decisión de negocio, no un
cálculo.

## 6. La «peor» pizza, mirada de cerca

Queda la segunda mitad del informe: eliminar la Brie Carre. Antes de firmar, mira qué clase de
producto es.

In [ ]:
brie = pizzas[pizzas["name"] == "The Brie Carre Pizza"]
ingreso_por_unidad = (por_ingresos / por_unidades).sort_values(ascending=False)

print(f"unidades en el año        : {brie['quantity'].sum():,}   (puesto 32 de 32)")
print(f"ingresos en el año        : {brie['ingreso'].sum():,.2f}   (puesto 32 de 32)")
print(f"precio                    : {brie['price'].unique()[0]}   (tamaños: {', '.join(brie['size'].unique())})")
print()
print("INGRESO POR UNIDAD VENDIDA (top 3 del menú)")
print(ingreso_por_unidad.head(3).round(2).to_string())
print()
print(f"puesto de la Brie Carre por ingreso por unidad: "
      f"{list(ingreso_por_unidad.index).index('The Brie Carre Pizza') + 1} de 32")

Última en unidades, última en ingresos… y **primera en ingreso por unidad**: cada Brie Carre que sale
del horno deja 23,65, tres dólares y medio más que la siguiente del menú (la Greek, con 20,04). Es la
pizza más cara de la carta y solo se vende en un tamaño: no es un fracaso de la Classic Deluxe, es
otro producto, jugando otro juego. ¿Se elimina? Con estos datos no lo sabes: para esa decisión falta
el **margen** —cuánto cuesta producirla—, y ese dato no está en la tabla. Decir *«esta pregunta no se
contesta con estos datos»* también es parte del análisis, y es la frase que separa un informe honesto
de uno que solo parece completo.

## 7. Un gráfico

Un gráfico bien hecho no ilustra el análisis: **es** el análisis. La regla del curso, desde hoy: el
título lleva la conclusión, no el nombre del eje.

In [ ]:
top10 = por_ingresos.head(10)
colores = ["#DD8452" if n == "The Classic Deluxe Pizza" else "#4C72B0" for n in top10.index]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(top10.index.str.replace("The ", "").str.replace(" Pizza", ""), top10.values, color=colores)
ax.invert_yaxis()
ax.set_title("La estrella depende de la métrica: Thai Chicken lidera los ingresos\n"
             "y la Classic Deluxe (naranja), primera en unidades, cae al cuarto puesto")
ax.set_xlabel("ingresos del año")
for y, valor in enumerate(top10.values):
    ax.text(valor + 300, y, f"{valor:,.0f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ✅ Comprobación 1 · la base cargó completa y el ingreso está bien construido
assert len(pizzas) == 48_620, \
    f"Deberías tener 48 620 líneas de pedido y tienes {len(pizzas):,}: ¿filtraste algo sin querer?"
assert pizzas["Factura"].nunique() == 21_350, \
    f"Deberían ser 21 350 pedidos y cuentas {pizzas['Factura'].nunique():,}: revisa que no hayas alterado la columna Factura"
assert abs(pizzas["ingreso"].sum() - 817_860.05) < 0.01, \
    "Tus ingresos no dan 817 860,05: la columna ingreso debe ser quantity × price, sin filtros"

print(f"Comprobación 1 superada ✓  {len(pizzas):,} líneas, {pizzas['Factura'].nunique():,} pedidos "
      f"y {pizzas['ingreso'].sum():,.2f} de ingresos")

## 8. Protocolo de uso de IA generativa

En este curso el asistente se usa desde hoy y para todo. Lo que se califica no es el código que
devuelve sino **la calidad del encargo y el rigor de la verificación**. El protocolo tiene tres reglas:

1. **Puedes pedir** código, explicaciones, alternativas y crítica de tu propio razonamiento.
2. **Tienes que verificar** todo número que salga del asistente contra una ejecución tuya. Si no puedes
   verificarlo, no puedes usarlo.
3. **Tienes que registrar** el prompt final y una línea sobre qué comprobaste. Eso es la bitácora.

La bitácora se entrega con cada cuaderno. Esta es la plantilla: complétala hoy con lo que le preguntes
al asistente durante la sesión.

In [ ]:
bitacora = pd.DataFrame(columns=[
    "fecha", "objetivo", "prompt_final", "que_devolvio",
    "como_lo_verifique", "veredicto",
])

def registrar(objetivo, prompt_final, que_devolvio, como_lo_verifique, veredicto):
    """Añade una fila a la bitácora. veredicto: 'acepto', 'corrijo' o 'descarto'."""
    global bitacora
    fila = pd.DataFrame([{
        "fecha": pd.Timestamp.today().date(),
        "objetivo": objetivo,
        "prompt_final": prompt_final,
        "que_devolvio": que_devolvio,
        "como_lo_verifique": como_lo_verifique,
        "veredicto": veredicto,
    }])
    bitacora = pd.concat([bitacora, fila], ignore_index=True)
    return bitacora

# Ejemplo resuelto: así se ve una fila bien escrita.
registrar(
    objetivo="Decidir qué métrica define a la 'pizza estrella' antes de calcular el ranking",
    prompt_final="Tengo un DataFrame con una fila por línea de pedido de una pizzería (columnas "
                 "quantity, price, name). Propón tres métricas alternativas para elegir la 'mejor' "
                 "pizza y explica qué decisión de negocio respalda cada una.",
    que_devolvio="Unidades vendidas, ingreso total e ingreso por unidad; margen si hubiera costos",
    como_lo_verifique="Calculé los tres rankings y comprobé que el número uno cambia según la métrica",
    veredicto="acepto",
)

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: llama a registrar(...) tres veces, una por cada prompt que uses hoy.
# Al menos uno de los tres debe terminar en veredicto="corrijo" o "descarto":
# si aceptas todo lo que te devuelve el asistente, no lo estás verificando.

bitacora

### 🌶️ Ejercicio 1 — Guiado

El informe comparó pizzas. Repite exactamente la misma trampa y la misma corrección con
**`category`** (Classic, Chicken, Supreme, Veggie): primero el ranking de categorías por unidades,
después por ingresos. Escribe en una frase qué cambia y qué no.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: pizzas.groupby("category")["quantity"].sum().sort_values(ascending=False)
# Pista 2: lo mismo con "ingreso" en lugar de "quantity"
# Pista 3: aquí, a diferencia de las pizzas, el líder NO cambia. Ahora divide ingresos
#          entre unidades por categoría: ¿qué categoría deja más por pizza vendida y
#          por qué esa tercera métrica sí cuenta otra historia?

### 🔥 Desafío

Construye la tabla completa de decisión del menú: las 32 pizzas con sus tres métricas —unidades,
ingresos e ingreso por unidad— y el **puesto** que ocupa cada pizza en cada ranking. Encuentra la
pizza que más puestos salta entre el ranking de unidades y el de ingresos, y escribe, en dos frases,
qué le recomendarías al dueño sobre ella y qué dato te falta para firmar la recomendación.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: arma un DataFrame con por_unidades, por_ingresos e ingreso_por_unidad
# Pista 2: .rank(ascending=False) convierte cada métrica en un puesto
# Pista 3: la diferencia absoluta entre dos columnas de puestos te da el salto

### 🎯 Reto en clase (15 min)

En parejas. Tomen la conclusión del informe («retirar la Brie Carre, promocionar la Classic Deluxe»)
y escriban **la versión corregida en tres frases**, con una cifra en cada una, listas para un dueño
que no va a ver su código. Después pídanle al asistente de IA que la ataque como si fuera el pizzero
que inventó la Brie Carre y defiende su receta, y registren el intercambio en la bitácora.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: no necesitas código nuevo, necesitas elegir tres números de los que ya calculaste.
# Escríbelos aquí como print() para que queden en el cuaderno entregado.

## La trampa de hoy

⚠️ **El ranking engañoso: contestar «cuál es la mejor» sin acordar antes qué significa «mejor».** El
informe del administrador se hizo con datos correctos, software correcto y aritmética correcta.
Ninguna versión más nueva de pandas y ningún modelo más grande lo habrían salvado: el error estaba en
la métrica, no en el cálculo. Los dos rankings lado a lado.

In [ ]:
comparacion = pd.DataFrame({
    "unidades": por_unidades,
    "ingresos": por_ingresos,
})
comparacion["puesto por unidades"] = comparacion["unidades"].rank(ascending=False).astype(int)
comparacion["puesto por ingresos"] = comparacion["ingresos"].rank(ascending=False).astype(int)
comparacion["salto"] = comparacion["puesto por unidades"] - comparacion["puesto por ingresos"]

print("Mejor pizza según unidades :", comparacion["unidades"].idxmax())
print("Mejor pizza según ingresos :", comparacion["ingresos"].idxmax())
print()
comparacion.sort_values("puesto por unidades").head(8)

In [ ]:
# ✅ Comprobación 2 · la trampa de hoy queda demostrada con estos dos hechos
assert por_unidades.idxmax() == "The Classic Deluxe Pizza" and por_unidades.max() == 2_453, \
    "El ranking por unidades debería coronar a la Classic Deluxe con 2 453: revisa el groupby por name"
assert por_ingresos.idxmax() == "The Thai Chicken Pizza" and abs(por_ingresos.max() - 43_434.25) < 0.01, \
    "El ranking por ingresos debería coronar a la Thai Chicken con 43 434,25: revisa que ingreso = quantity × price"

print("Comprobación 2 superada ✓  dos métricas, dos estrellas distintas: esa es la lección de hoy")

Las dos columnas salen del mismo archivo, de la misma máquina y del mismo lenguaje. Lo único que
cambia es la métrica. Si el dueño hubiera preguntado *«¿mejor en qué?»* —la pregunta que abre la
semana 2— el informe no habría salido de la cocina.

## Entregable

Sube `lab_01_apellido.ipynb` con:

- El cuaderno ejecutado de arriba a abajo, sin errores y sin celdas salteadas.
- Los tres ejercicios resueltos, cada uno con su celda de código y una frase de interpretación.
- La bitácora con **al menos tres filas propias**, una de ellas con veredicto `corrijo` o `descarto`.
- Al final, en una celda de texto: en qué celda de este cuaderno la conclusión del informe se da
  vuelta. Una sola celda, señálala por su contenido.

## Para tu equipo

- Firmen el contrato de grupo y abran el repositorio compartido esta misma semana. La bitácora de
  prompts vive ahí desde hoy, no desde la semana en que se entrega el proyecto.
- Elijan una empresa cuyos datos puedan conseguir de verdad. Un caso brillante sin datos se convierte
  en un ensayo, y este curso no califica ensayos.
- Busquen en su empresa el equivalente del informe de hoy: un ranking que se reporta con la métrica
  cómoda (unidades, visitas, descargas) cuando la decisión depende de otra (ingresos, margen,
  retención). Lo van a encontrar. Anótenlo: es el punto de partida del proyecto integrador.